In [2]:
import pandas as pd

In [2]:
# run ImmunoMTL first:
# python3 eval_ImmunoMTL.py
# python3 eval_ImmunoMTL_shuffle.py
# python3 eval_ImmunoSTL.py

In [13]:
import os
import pandas as pd

def load_and_clean_predictions(directory=".", out="."):
    prediction_dfs = {}

    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            filepath = os.path.join(directory, filename)
            try:
                df = pd.read_csv(filepath)
                # Drop columns if they exist
                df = df.drop(columns=["ImmunoMTL_score", "MMS_Cluster"], errors="ignore")

                prediction_dfs[filename] = df
                cleaned_path = os.path.join(out, filename)
                df.to_csv(cleaned_path, index=False)
            except Exception as e:
                print(f"[ERROR] Failed to load {filename}: {e}")
    return prediction_dfs

# === Usage ===
cleaned_predictions = load_and_clean_predictions("../pred_results/immunomtl", out = "../pred_results")

# Preview an example
for name, df in cleaned_predictions.items():
    print(f"\n{name}")
    print(df.shape)
    print(df.head())


mRNA.csv
(222, 3)
       Peptide          MHC  Label
0  ITVNASRPQPF  HLA-A*32:01      0
1     RLAAAVRF  HLA-A*32:01      0
2  AVMHLDHSDTI  HLA-A*32:01      0
3  VSYANSCANPV  HLA-C*17:01      0
4   MSAEVNLAGL  HLA-C*17:01      1

zero1.csv
(1172, 3)
     Peptide          MHC  Label
0  FYVFDEPLL  HLA-A*24:07      0
1  RYARTIFNF  HLA-A*24:07      0
2  IFKDSTMHI  HLA-A*24:07      0
3  LFYVYYNLF  HLA-A*24:07      0
4  VYYNLFLLF  HLA-A*24:07      0

BenchmarkSet.csv
(2495, 3)
     Peptide          MHC  Label
0   FLKEKGGL  HLA-B*08:01      1
1  ELRRKMMYM  HLA-B*08:01      1
2  QIKVRVDMV  HLA-B*08:01      1
3  FLRGRAYGL  HLA-B*08:01      1
4  HSKKKCDEL  HLA-B*08:01      1

zero2.csv
(77, 3)
      Peptide          MHC  Label
0    RRFFPYYV  HLA-B*27:03      1
1  FLPSDFFPSV  HLA-A*02:17      1
2   WLSLLVPFV  HLA-A*02:17      1
3   YLPGVIAAI  HLA-A*02:17      1
4    GPISGHVL  HLA-B*81:01      1


In [14]:
#bigMHC 
#conda env: pbert
# pre install bigMHC
# 2135  python3 predict.py -o ../../ImmunoMTL/pred_results/bigmhc/BenchmarkSet_bigmhc.csv -i ../../ImmunoMTL/pred_results/BenchmarkSet.csv -m IM -d all -a 1 -p 0 
# 2136  python3 predict.py -o ../../ImmunoMTL/pred_results/bigmhc/mRNA_bigmhc.csv -i ../../ImmunoMTL/pred_results/mRNA.csv -m IM -d all -a 1 -p 0 
# 2139  python3 predict.py -o ../../ImmunoMTL/pred_results/bigmhc/zero1_bigmhc.csv -i ../../ImmunoMTL/pred_results/zero1.csv -m IM -d all -a 1 -p 0 
# 2140  python3 predict.py -o ../../ImmunoMTL/pred_results/bigmhc/zero2_bigmhc.csv -i ../../ImmunoMTL/pred_results/zero2.csv -m IM -d all -a 1 -p 0


In [15]:
#for munis

processed = {}
for name, df in cleaned_predictions.items():
    try:
        # Rename columns
        df = df.rename(columns={"Peptide": "pep", "MHC": "mhc"})

        # Create 'left' and 'right' columns
        df.insert(loc=df.columns.get_loc("mhc") + 1, column="left", value="")
        df.insert(loc=df.columns.get_loc("left") + 1, column="right", value="")

        processed[name] = df
        print(f"Processed: {name}")
        print(df.head())
        df.to_csv(f"../pred_results/munis/{name}")
    except Exception as e:
        print(f"Failed to process {name}: {e}")


Processed: mRNA.csv
           pep          mhc left right  Label
0  ITVNASRPQPF  HLA-A*32:01                 0
1     RLAAAVRF  HLA-A*32:01                 0
2  AVMHLDHSDTI  HLA-A*32:01                 0
3  VSYANSCANPV  HLA-C*17:01                 0
4   MSAEVNLAGL  HLA-C*17:01                 1
Processed: zero1.csv
         pep          mhc left right  Label
0  FYVFDEPLL  HLA-A*24:07                 0
1  RYARTIFNF  HLA-A*24:07                 0
2  IFKDSTMHI  HLA-A*24:07                 0
3  LFYVYYNLF  HLA-A*24:07                 0
4  VYYNLFLLF  HLA-A*24:07                 0
Processed: BenchmarkSet.csv
         pep          mhc left right  Label
0   FLKEKGGL  HLA-B*08:01                 1
1  ELRRKMMYM  HLA-B*08:01                 1
2  QIKVRVDMV  HLA-B*08:01                 1
3  FLRGRAYGL  HLA-B*08:01                 1
4  HSKKKCDEL  HLA-B*08:01                 1
Processed: zero2.csv
          pep          mhc left right  Label
0    RRFFPYYV  HLA-B*27:03                 1
1  FLPSDFFPSV  H

In [16]:
#munis
#conda env: munis
# pre install munis
# 2072  python predict.py --outdir ../ImmunoMTL/pred_results/munis/ --peptides ../ImmunoMTL/pred_results/munis/BenchmarkSet.csv 
# 2073  python predict.py --outdir ../ImmunoMTL/pred_results/munis/ --peptides ../ImmunoMTL/pred_results/munis/mRNA.csv 
# 2075  python predict.py --outdir ../ImmunoMTL/pred_results/munis/ --peptides ../ImmunoMTL/pred_results/munis/zero1.csv 
# 2076  python predict.py --outdir ../ImmunoMTL/pred_results/munis/ --peptides ../ImmunoMTL/pred_results/munis/zero2.csv


In [25]:
#PRIME
#cd bin 
#python3 PRIME_predict.py --input ~/bin/ImmunoMTL/pred_results/zero1.csv --hla MHC --pep Peptide --l Label --out ~/bin/ImmunoMTL/pred_results/prime2/zero1_prime.csv
#python3 PRIME_predict.py --input ~/bin/ImmunoMTL/pred_results/zero2.csv --hla MHC --pep Peptide --l Label --out ~/bin/ImmunoMTL/pred_results/prime2/zero2_prime.csv
#python3 PRIME_predict.py --input ~/bin/ImmunoMTL/pred_results/BenchmarkSet.csv --hla MHC --pep Peptide --l Label --out ~/bin/ImmunoMTL/pred_results/prime2/BenchmarkSet_prime.csv
#python3 PRIME_predict.py --input ~/bin/ImmunoMTL/pred_results/mRNA.csv --hla MHC --pep Peptide --l Label --out ~/bin/ImmunoMTL/pred_results/prime2/mRNA_prime.csv

In [ ]:
#mhcflurry-predict ../pred_results/immunomtl/BenchmarkSet.csv --allele-column MHC --peptide-column Peptide --out ../pred_results/mhcflurry/BenchmarkSet_mhcflurry.csv
#mhcflurry-predict ../pred_results/immunomtl/mRNA.csv --allele-column MHC --peptide-column Peptide --out ../pred_results/mhcflurry/mRNA_mhcflurry.csv
#mhcflurry-predict ../pred_results/immunomtl/zero1.csv --allele-column MHC --peptide-column Peptide --out ../pred_results/mhcflurry/zero1_mhcflurry.csv
#mhcflurry-predict ../pred_results/immunomtl/zero2.csv --allele-column MHC --peptide-column Peptide --out ../pred_results/mhcflurry/zero2_mhcflurry.csv

In [26]:
#netMHCpan
#python3 BA_predict.py --input ~/bin/ImmunoMTL/pred_results/BenchmarkSet.csv --hla MHC --pep Peptide --l Label --out ~/bin/ImmunoMTL/pred_results/netMHCpan/BenchmarkSet_netMHCpan.csv
#python3 BA_predict.py --input ~/bin/ImmunoMTL/pred_results/mRNA.csv --hla MHC --pep Peptide --l Label --out ~/bin/ImmunoMTL/pred_results/netMHCpan/mRNA_netMHCpan.csv
#python3 BA_predict.py --input ~/bin/ImmunoMTL/pred_results/zero1.csv --hla MHC --pep Peptide --l Label --out ~/bin/ImmunoMTL/pred_results/netMHCpan/zero1_netMHCpan.csv
#python3 BA_predict.py --input ~/bin/ImmunoMTL/pred_results/zero2.csv --hla MHC --pep Peptide --l Label --out ~/bin/ImmunoMTL/pred_results/netMHCpan/zero2_netMHCpan.csv

In [11]:
import os
import pandas as pd
import numpy as np

# === CONFIG ===
folders = {
    "immunostl": "../pred_results/immunostl",
    "munis": "../pred_results/munis",
    "deepimmuno": "../pred_results/deepimmuno",
    "bigmhc": "../pred_results/bigmhc",
    "prime2": "../pred_results/prime2",
    "shuffle": "../pred_results/immunomtl_shuffle",
    "netMHCpan": "../pred_results/netMHCpan",
    "mhcflurry": "../pred_results/mhcflurry",
}

external_tools = {
    "immunostl": ("immunostl", "", "Predicted Score"),
    "BigMHC_IM": ("bigmhc", "_bigmhc", "BigMHC_IM"),
    "munis": ("munis", "_munis_predictions", "score"),
    "PRIME_score": ("prime2", "_prime", "PRIME_score"),
    "ImmunoMTL_shuffle": ("shuffle", "", "Predicted Score"),
    "netMHCpan_ELscore": ("netMHCpan", "_netMHCpan", "EL-score"),
    "MHCflurry_presentation_score": ("mhcflurry", "_mhcflurry", "mhcflurry_presentation_score"),
}

datasets = ["zero1", "zero2", "BenchmarkSet"]
output_dir = "../analysis/"
os.makedirs(output_dir, exist_ok=True)

for dataset in datasets:
    print(f"[INFO] Processing {dataset}")
    df_base = pd.read_csv(os.path.join("../pred_results/immunomtl", f"{dataset}.csv"))

    # Ensure necessary columns
    if "ImmunoMTL_score" not in df_base.columns:
        df_base["ImmunoMTL_score"] = np.nan
    if "Peptide" not in df_base.columns or "MHC" not in df_base.columns:
        raise ValueError("Input files must contain 'Peptide' and 'MHC' columns")

    df_base["pMHC"] = df_base["Peptide"] + df_base["MHC"]

    # Add scores from external tools
    for key, (folder_key, suffix, col) in external_tools.items():
        file_path = os.path.join(folders[folder_key], f"{dataset}{suffix}.csv")
        if os.path.exists(file_path):
            try:
                ext_df = pd.read_csv(file_path)

                # Handle special column names
                if "pep" in ext_df.columns and "mhc" in ext_df.columns:
                    ext_df["pMHC"] = ext_df["pep"] + ext_df["mhc"]
                elif "Peptide" in ext_df.columns and "MHC" in ext_df.columns:
                    ext_df["pMHC"] = ext_df["Peptide"] + ext_df["MHC"]
                else:
                    print(f"[WARN] {key} missing peptide/MHC columns for pMHC construction")
                    df_base[key] = np.nan
                    continue

                # Handle score/label column renaming if needed
                # Always rename the score column to match the key (tool name)
                if key == "munis" and "score" in ext_df.columns:
                    ext_df = ext_df.rename(columns={"score": key})
                elif key == "BigMHC_IM" and "BigMHC_IM" in ext_df.columns:
                    ext_df = ext_df.rename(columns={"BigMHC_IM": key})
                elif key == "immunostl" and "Predicted Score" in ext_df.columns:
                    ext_df = ext_df.rename(columns={"Predicted Score": key})
                elif col in ext_df.columns:
                    ext_df = ext_df.rename(columns={col: key})
                else:
                    print(f"[WARN] Cannot find suitable column to rename for {key}")
                    df_base[key] = np.nan
                    continue
                df_base = df_base.merge(ext_df[["pMHC", key]], on="pMHC", how="left")
            except Exception as e:
                print(f"[WARN] {key} column issue in {file_path}: {e}")
                df_base[key] = np.nan
        else:
            print(f"[INFO] {key} not found for {dataset}")
            df_base[key] = np.nan

    # Save individual file
    print(df_base.shape)
    output_path = os.path.join(output_dir, f"{dataset}_pred.csv")
    df_base.to_csv(output_path, index=False)
    print(f"[INFO] Saved to {output_path}")

[INFO] Processing zero1
[INFO] ImmunoMTL_shuffle not found for zero1
(1172, 13)
[INFO] Saved to ../analysis/zero1_pred.csv
[INFO] Processing zero2
[INFO] ImmunoMTL_shuffle not found for zero2
(77, 13)
[INFO] Saved to ../analysis/zero2_pred.csv
[INFO] Processing BenchmarkSet
(2495, 13)
[INFO] Saved to ../analysis/BenchmarkSet_pred.csv


In [9]:
#Vali pool
mRNA_result = pd.read_csv("../pred_results/immunomtl/mRNA.csv")

mRNA_result["pMHC"] = mRNA_result["Peptide"]+mRNA_result["MHC"]

mRNA_result_bigmhc = pd.read_csv("../pred_results/bigmhc/mRNA_bigmhc.csv")
mRNA_result["BigMHC_IM"] = mRNA_result_bigmhc["BigMHC_IM"]

mRNA_result_prime = pd.read_csv("../pred_results/prime2/mRNA_prime.csv")
mRNA_result["PRIME_score"] = mRNA_result_prime["PRIME_score"]
mRNA_result["PRIME_rank"] = mRNA_result_prime["PRIME_rank"]

mRNA_result_munis = pd.read_csv("../pred_results/munis/mRNA_munis_predictions.csv")
mRNA_result["munis"] = mRNA_result_munis["score"]

mRNA_result_netMHCpan = pd.read_csv("../pred_results/netMHCpan/mRNA_netMHCpan.csv")
mRNA_result["netMHCpan_score"] = mRNA_result_netMHCpan["EL-score"]
mRNA_result["netMHCpan_rank"] = mRNA_result_netMHCpan["EL_Rank"]

mRNA_result_mhcflurry = pd.read_csv("../pred_results/mhcflurry/mRNA_mhcflurry.csv")
mRNA_result["mhcflurry_presentation_score"] = mRNA_result_mhcflurry["mhcflurry_presentation_score"]
mRNA_result["mhcflurry_presentation_rank"] = mRNA_result_mhcflurry["mhcflurry_presentation_percentile"]

#mRNA_result.to_csv("../analysis/mRNA_rank.csv")

In [10]:
patient_info = pd.read_csv("../data/mRNAvaccine_pID.csv")
patient_info["pMHC"] = patient_info["Peptide"]+patient_info["MHC"]


mRNA_result = pd.merge(patient_info[["pMHC", "patientID"]], mRNA_result, on="pMHC", how="left")
mRNA_result.to_csv("../analysis/mRNA_pred.csv", index = False)
mRNA_result

,pMHC,patientID,Peptide,MHC,MMS_Cluster,Label,ImmunoMTL_score,BigMHC_IM,PRIME_score,PRIME_rank,munis,netMHCpan_score,netMHCpan_rank,mhcflurry_presentation_score,mhcflurry_presentation_rank
0,ITVNASRPQPFHLA-A*32:01,1,ITVNASRPQPF,HLA-A*32:01,0.0,0,9.296952e-01,0.000514,0.002020,19.170,0.151489,0.0126,4.6814,0.029584,8.269212
1,RLAAAVRFHLA-A*32:01,5,RLAAAVRF,HLA-A*32:01,0.0,0,2.207086e-08,0.006755,0.004967,8.816,0.271729,0.0734,1.5100,0.121985,2.557500
2,AVMHLDHSDTIHLA-A*32:01,5,AVMHLDHSDTI,HLA-A*32:01,0.0,0,4.035173e-01,0.000256,0.014185,3.145,0.095215,0.0201,3.6059,0.028207,8.545326
3,TEYKLVVVGAVHLA-B*41:01,1,TEYKLVVVGAV,HLA-B*41:01,1.0,0,9.999158e-01,0.005845,0.001803,20.529,0.219482,0.0498,1.9947,0.310056,1.330625
4,CDSHGLGKNPIHLA-B*41:01,1,CDSHGLGKNPI,HLA-B*41:01,1.0,0,2.249859e-06,0.047019,0.000681,40.298,0.005642,0.0000,45.3333,0.027011,9.155353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,LADLLNPIHLA-C*08:02,29,LADLLNPI,HLA-C*08:02,3.0,1,9.113949e-01,0.280507,0.014621,2.698,0.526855,0.1393,0.7491,0.631520,0.558315
218,MSCRVGEELHLA-A*68:02,29,MSCRVGEEL,HLA-A*68:02,0.0,0,4.563048e-04,0.542159,0.023657,1.844,0.493896,0.0467,2.8674,0.115379,2.646576
219,DTYVPTSDYKHLA-A*68:02,29,DTYVPTSDYK,HLA-A*68:02,0.0,0,2.675578e-05,0.644923,0.039359,1.001,0.285400,0.0139,5.8400,0.170581,2.042935
220,THESDLPPSDKHLA-A*68:02,29,THESDLPPSDK,HLA-A*68:02,0.0,0,9.537110e-01,0.212156,0.000548,44.581,0.312988,0.0000,55.0000,0.013242,17.305598
